# Capítulo 6 — Aprendizado de Máquina Estatístico

**Livro:** começa na página **269** do PDF.

📗 **Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
rng = np.random.default_rng(42)
print("Pronto. pandas", pd.__version__)

## Padronização (Normalização, Escores Z) · PDF p. 276

#### 🧪 Exercício: KNN com e sem padronização

Nos dados de vinhos, compare a acurácia (validação cruzada de 5 lotes) do KNN (k = 5) com as variáveis brutas e padronizadas (`StandardScaler` num `make_pipeline`).

In [ ]:
from sklearn.datasets import load_wine
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
X, y = load_wine(return_X_y=True)
print(round(cross_val_score(KNeighborsClassifier(5), X, y, cv=5).mean(), 3),
      round(cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(5)), X, y, cv=5).mean(), 3))

## Escolhendo K · PDF p. 278

#### 🧪 Exercício: Escolhendo k por validação cruzada

Use `GridSearchCV` para escolher k entre 1 e 30 no KNN padronizado dos vinhos e mostre o melhor k e a acurácia.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
X, y = load_wine(return_X_y=True)
g = GridSearchCV(make_pipeline(StandardScaler(), KNeighborsClassifier()), {"kneighborsclassifier__n_neighbors": range(1, 31)}, cv=5).fit(X, y)
print(g.best_params_, round(g.best_score_, 3))

## Medindo Homogeneidade ou Impureza · PDF p. 286

#### 🧪 Exercício: Gini e entropia de uma divisão

Escreva `gini(p)` e `entropia(p)` para uma proporção p de positivos. Para um nó com 40 positivos e 60 negativos dividido em (30+, 10−) e (10+, 50−), calcule a impureza antes e a impureza ponderada depois.

In [ ]:
def gini(p): return 2 * p * (1 - p)
def entropia(p): return 0 if p in (0, 1) else -(p * np.log2(p) + (1 - p) * np.log2(1 - p))
for f in (gini, entropia):
    antes = f(40 / 100); depois = 40 / 100 * f(30 / 40) + 60 / 100 * f(10 / 60)
    print(f.__name__, round(antes, 3), round(depois, 3))

## Floresta Aleatória · PDF p. 295

#### 🧪 Exercício: Árvore × floresta

Nos dados de câncer de mama, compare a acurácia (validação cruzada) de uma árvore de decisão e de uma floresta aleatória com 300 árvores.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
X, y = load_breast_cancer(return_X_y=True)
for m in (DecisionTreeClassifier(random_state=0), RandomForestClassifier(300, random_state=0)):
    print(type(m).__name__, round(cross_val_score(m, X, y, cv=5).mean(), 3))

## Importância da Variável · PDF p. 298

#### 🧪 Exercício: Importância por permutação

Treine uma floresta nos dados de câncer de mama e mostre as 5 variáveis mais importantes por `permutation_importance` no conjunto de teste.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
d = load_breast_cancer(as_frame=True)
Xtr, Xte, ytr, yte = train_test_split(d.data, d.target, test_size=0.3, random_state=0)
rf = RandomForestClassifier(300, random_state=0).fit(Xtr, ytr)
imp = permutation_importance(rf, Xte, yte, n_repeats=10, random_state=0)
print(pd.Series(imp.importances_mean, d.data.columns).sort_values(ascending=False).head(5).round(4))

## XGBoost · PDF p. 305

#### 🧪 Exercício: Boosting com HistGradientBoosting

Use `HistGradientBoostingClassifier` (o *gradient boosting* do scikit-learn, parecido com o XGBoost) nos dados de câncer de mama e compare com a floresta. Depois teste `learning_rate` de 0,01, 0,1 e 0,5.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score
X, y = load_breast_cancer(return_X_y=True)
for lr in (0.01, 0.1, 0.5):
    print(lr, round(cross_val_score(HistGradientBoostingClassifier(learning_rate=lr, random_state=0), X, y, cv=5).mean(), 3))

## Hiperparâmetros e Validação Cruzada · PDF p. 310

#### 🧪 Exercício: Busca de hiperparâmetros

Com `RandomizedSearchCV` (20 combinações), ajuste `max_depth`, `learning_rate` e `max_iter` do HistGradientBoosting e mostre a melhor combinação.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
X, y = load_breast_cancer(return_X_y=True)
busca = RandomizedSearchCV(HistGradientBoostingClassifier(random_state=0),
    {"max_depth": [2, 3, 5, None], "learning_rate": [0.01, 0.05, 0.1, 0.3], "max_iter": [100, 200, 400]},
    n_iter=20, cv=5, random_state=0).fit(X, y)
print(busca.best_params_, round(busca.best_score_, 3))